# House Sale Data — Feature Engineering
This notebook loads the Baku real-estate listing dataset (`house_sale.csv`) and walks through:

1. Loading the data
2. A quick look at the relevant columns
3. Light cleaning of the fields needed for feature engineering
4. **Feature engineering** — creating new, meaningful columns from existing ones

The focus of this notebook is step 4, but steps 1–3 are included so the new features can be computed correctly.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

## 1. Load the data

In [2]:
df = pd.read_csv('house_sale.csv')
print(df.shape)
df.head(3)

(100775, 51)


,id_x,rel_url,estate_rel_url_x,datetime_scrape_x,price,currency_x,location,attributes,city_when,city,day_x,hour_x,repair,vip,featured,products_label,bill_of_sale,mortgage,img_url,id_y,estate_id,estate_rel_url_y,datetime_scrape_y,description,unit_price,total_price,currency_y,owner_name,owner_title,shop_name,shop_title,address,lat,lng,updated,views,day_y,hour_y,estate_details_id_x,Binanın növü,Kateqoriya,Mərtəbə,Otaq sayı,Sahə,Torpaq sahəsi,Təmir,Çıxarış,İpoteka,estate_details_id_y,estate_rel_url,extra_info
0,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/alqi-satqi?page=174,/items/4521724,2024-10-05 22:07:37.60613+00,499999.0,AZN,Səbail r.,"4 otaqlı, 145 m², 7/9 mərtəbə","Bakı, dünən 23:52",bakı,05.10.2024,23:52,Təmirli,vipped,featured,NaN,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,92e82ea2-e1f3-4c2d-a284-151efd99281e,5df36281-6dc6-4d5d-89a7-5fcfa86f7608,/items/4521724,2024-10-05 22:14:10.089116+00,"Səbail Rayonu, İzzət Nəbiyev küçəsi, Fəxri Xiy...",3 450 AZN/m²,499999.0,AZN,Kamran,mülkiyyətçi,NaN,NaN,İzzət Nəbiyev küç.,40.358817,49.824092,yeniləndi: dünən 23:52,1155,05.10.2024,23:52,92e82ea2-e1f3-4c2d-a284-151efd99281e,NaN,Köhnə tikili,7 / 9,4.0,145 m²,NaN,var,var,NaN,92e82ea2-e1f3-4c2d-a284-151efd99281e,/items/4521724,Şəhidlər xiyabanı * Dağüstü parkı * Səbail r.
1,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/alqi-satqi?page=250,/items/4669294,2024-10-05 22:07:37.60613+00,77000.0,AZN,Biləcəri q.,"4 otaqlı, 90 m²","Bakı, dünən 23:56",bakı,05.10.2024,23:56,Təmirli,NaN,NaN,NaN,NaN,NaN,https://bina.azstatic.com/uploads/f460x345/202...,505eaf81-6bc8-4094-9b00-aa82066548ee,883e20f0-8872-49a5-8b4b-63b8301b5f8f,/items/4669294,2024-10-05 22:14:10.089116+00,"Biləcəridə Abidəyə yaxin 91,92,202 saylı marşr...",NaN,77000.0,AZN,Dasinmaz Emlak,vasitəçi (agent),NaN,NaN,Biləcəri qəs.,40.420897,49.807035,yeniləndi: 04 oktyabr 2024,218,04.10.2024,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,NaN,Həyət evi/Bağ evi,NaN,4.0,90 m²,1.3 sot,var,yoxdur,NaN,505eaf81-6bc8-4094-9b00-aa82066548ee,/items/4669294,Binəqədi r.* Biləcəri q.
2,55c36fb1-a3af-476e-ba17-a81f6795be8d,/alqi-satqi?page=250,/items/4669293,2024-10-05 22:07:37.60613+00,92000.0,AZN,İnşaatçılar m.,"3 otaqlı, 60 m²","Bakı, dünən 23:55",bakı,05.10.2024,23:55,Təmirli,NaN,NaN,NaN,Çıxarış var,NaN,https://bina.azstatic.com/uploads/f460x345/202...,fa63b201-999d-43b5-a61a-778d9d79a6c6,55c36fb1-a3af-476e-ba17-a81f6795be8d,/items/4669293,2024-10-05 22:14:10.089116+00,Salam əleykum. \nİnşaatçılar metrosuna yaxın m...,NaN,92000.0,AZN,Məhəmməd,vasitəçi (agent),NaN,NaN,Mirzə Cabbar Məmmədzadə küç.,40.390293,49.802656,yeniləndi: 04 oktyabr 2024,190,04.10.2024,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,NaN,Həyət evi/Bağ evi,NaN,3.0,60 m²,0.1 sot,var,var,NaN,fa63b201-999d-43b5-a61a-778d9d79a6c6,/items/4669293,İnşaatçılar m.* Yasamal r.


## 2. Columns we'll use

The raw file has 51 columns (scraped listing metadata). For feature engineering we care mainly about:

| Column | Meaning |
|---|---|
| `price` | Listing price (AZN) |
| `Sahə` | Living area, e.g. `"145 m²"` (text) |
| `Torpaq sahəsi` | Land area, e.g. `"1.3 sot"` (text, houses/land only) |
| `Otaq sayı` | Number of rooms |
| `Mərtəbə` | Floor, e.g. `"7 / 9"` meaning floor 7 of a 9-floor building |
| `views` | Number of times the listing was viewed |
| `datetime_scrape_x` | When the listing was first scraped |
| `updated` | When the listing was last updated |
| `Təmir` | Renovation status (`var` = renovated, `yoxdur` = not) |

## 3. Light cleaning

We extract numeric values out of the text fields so we can do arithmetic on them.

In [3]:
# Living area: "145 m²" -> 145.0
df['area_sqm'] = df['Sahə'].str.extract(r'([\d.]+)').astype(float)

# Land area: "1.3 sot" -> 1.3  (1 sot = 100 m² in Azerbaijan)
df['land_sot'] = df['Torpaq sahəsi'].str.extract(r'([\d.]+)').astype(float)

# Floor: "7 / 9" -> current floor 7, total floors 9
floor_split = df['Mərtəbə'].str.split('/', expand=True)
df['floor_current'] = pd.to_numeric(floor_split[0], errors='coerce')
df['floor_total'] = pd.to_numeric(floor_split[1], errors='coerce')

# Dates
df['datetime_scrape_x'] = pd.to_datetime(df['datetime_scrape_x'], errors='coerce', utc=True)
df['updated'] = pd.to_datetime(df['updated'], errors='coerce', utc=True)

df[['price', 'area_sqm', 'land_sot', 'floor_current', 'floor_total', 'Otaq sayı', 'views']].describe()

/tmp/ipykernel_580/1196023102.py:14: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['updated'] = pd.to_datetime(df['updated'], errors='coerce', utc=True)


,price,area_sqm,land_sot,floor_current,floor_total,Otaq sayı,views
count,1.007750e+05,100775.000000,15571.000000,75996.000000,75996.000000,91363.000000,100775.000000
mean,3.423557e+05,143.941045,11.572430,7.851782,13.555582,3.133468,700.065185
std,2.042627e+06,1004.402694,298.500058,4.886824,5.453310,1.372981,1680.672574
min,1.100000e+01,0.200000,0.100000,1.000000,1.000000,1.000000,23.000000
25%,1.450000e+05,65.350000,2.500000,4.000000,9.000000,2.000000,99.000000
50%,2.180000e+05,100.000000,4.000000,7.000000,16.000000,3.000000,254.000000
75%,3.380000e+05,145.000000,6.500000,11.000000,17.000000,4.000000,680.000000
max,6.000000e+08,200000.000000,36000.000000,27.000000,33.000000,20.000000,113458.000000


## 4. Feature engineering

We now create new columns that carry more analytical meaning than the raw fields on their own.

### Feature 1 — `price_per_sqm`
Raw `price` mixes together the effect of size and location/quality. Dividing price by living area gives a **normalized price** that is comparable across listings of very different sizes — the standard metric used to judge whether a property is over- or under-priced for its area.

### Feature 2 — `floor_ratio` and `floor_position`
`floor_current` and `floor_total` on their own don't tell you much. Their **ratio** (how high up the building the unit sits, from 0=ground to 1=top) is a meaningful predictor of price, since ground floors and top floors are usually valued differently from middle floors. We also derive a categorical flag (`ground`, `top`, `middle`) which is easier to use in plots/models than a raw ratio.

### Feature 3 (bonus) — `room_size_sqm`
Living area divided by number of rooms approximates the **average room size**, which captures layout quality independent of the total footprint (a 100 m² 2-room flat feels very different from a 100 m² 5-room flat).

### Feature 4 (bonus) — `listing_age_days` and `views_per_day`
The gap between first scrape and last update tells us how long a listing has been active. Combined with `views`, we get a **popularity rate** (views per day) instead of a raw, time-unadjusted view count — this makes listings scraped at different times comparable.

In [4]:
# --- Feature 1: normalized price ---
df['price_per_sqm'] = df['price'] / df['area_sqm']

# --- Feature 2: floor position ---
df['floor_ratio'] = df['floor_current'] / df['floor_total']

def floor_position(row):
    if pd.isna(row['floor_current']) or pd.isna(row['floor_total']):
        return np.nan
    if row['floor_current'] == 1:
        return 'ground'
    if row['floor_current'] == row['floor_total']:
        return 'top'
    return 'middle'

df['floor_position'] = df.apply(floor_position, axis=1)

# --- Feature 3: average room size ---
df['room_size_sqm'] = df['area_sqm'] / df['Otaq sayı']

# --- Feature 4: listing age & popularity rate ---
df['listing_age_days'] = (df['updated'] - df['datetime_scrape_x']).dt.total_seconds() / 86400
df['listing_age_days'] = df['listing_age_days'].clip(lower=0)  # guard against negative gaps
df['views_per_day'] = df['views'] / df['listing_age_days'].replace(0, np.nan)

new_cols = ['price_per_sqm', 'floor_ratio', 'floor_position', 'room_size_sqm',
            'listing_age_days', 'views_per_day']
df[new_cols].describe(include='all')

,price_per_sqm,floor_ratio,floor_position,room_size_sqm,listing_age_days,views_per_day
count,1.007750e+05,75996.000000,75996,91363.000000,0.0,0.0
unique,NaN,NaN,3,NaN,NaN,NaN
top,NaN,NaN,middle,NaN,NaN,NaN
freq,NaN,NaN,66144,NaN,NaN,NaN
mean,3.544907e+03,0.590130,NaN,37.978576,NaN,NaN
std,1.497575e+04,0.267371,NaN,14.746079,NaN,NaN
min,2.500000e-02,0.030303,NaN,0.250000,NaN,NaN
25%,1.800000e+03,0.368421,NaN,28.750000,NaN,NaN
50%,2.293478e+03,0.600000,NaN,35.500000,NaN,NaN
75%,2.850000e+03,0.818182,NaN,45.000000,NaN,NaN


## 5. Sanity check — before vs. after

In [5]:
cols_to_show = ['price', 'area_sqm', 'price_per_sqm',
                'floor_current', 'floor_total', 'floor_ratio', 'floor_position',
                'Otaq sayı', 'room_size_sqm',
                'views', 'listing_age_days', 'views_per_day']
df[cols_to_show].head(10)

,price,area_sqm,price_per_sqm,floor_current,floor_total,floor_ratio,floor_position,Otaq sayı,room_size_sqm,views,listing_age_days,views_per_day
0,499999.0,145.0,3448.268966,7.0,9.0,0.777778,middle,4.0,36.250000,1155,NaN,NaN
1,77000.0,90.0,855.555556,NaN,NaN,NaN,NaN,4.0,22.500000,218,NaN,NaN
2,92000.0,60.0,1533.333333,NaN,NaN,NaN,NaN,3.0,20.000000,190,NaN,NaN
3,95000.0,130.0,730.769231,NaN,NaN,NaN,NaN,NaN,NaN,1314,NaN,NaN
4,220000.0,100.0,2200.000000,15.0,16.0,0.937500,middle,3.0,33.333333,240,NaN,NaN
5,650000.0,130.0,5000.000000,3.0,4.0,0.750000,middle,4.0,32.500000,886,NaN,NaN
6,259000.0,70.0,3700.000000,2.0,5.0,0.400000,middle,3.0,23.333333,2313,NaN,NaN
7,279000.0,153.0,1823.529412,3.0,18.0,0.166667,middle,3.0,51.000000,3119,NaN,NaN
8,3000000.0,485.0,6185.567010,NaN,NaN,NaN,NaN,NaN,NaN,1316,NaN,NaN
9,262000.0,104.0,2519.230769,11.0,13.0,0.846154,middle,3.0,34.666667,103,NaN,NaN


## 6. Save the enriched dataset

In [6]:
df.to_csv('house_sale_features.csv', index=False)
print('Saved house_sale_features.csv with', df.shape[1], 'columns (', df.shape[1]-51, 'new).')

Saved house_sale_features.csv with 61 columns ( 10 new).
